In [20]:
#load 
from pathlib import Path
import subprocess
import pandas as pd
import tempfile
import os

OpenSMILE, intensity 3 dB

In [21]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations_pain" / "+3dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_3dB_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_3dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1005
[1/1005] Processing: p10085.LC.1.161.wav
[2/1005] Processing: p10085.LC.12.59.wav
[3/1005] Processing: p10085.LC.14.45.wav
[4/1005] Processing: p10085.LC.18.41.wav
[5/1005] Processing: p10085.LC.2.106.wav
[6/1005] Processing: p10085.LC.23.156.wav
[7/1005] Processing: p10085.LC.24.172.wav
[8/1005] Processing: p10085.LC.26.99999.wav
[9/1005] Processing: p10085.LC.3.60.wav
[10/1005] Processing: p10085.LC.30.132.wav
[11/1005] Processing: p10085.LC.31.99999.wav
[12/1005] Processing: p10085.LC.33.6.wav
[13/1005] Processing: p10085.LC.34.122.wav
[14/1005] Processing: p10085.LC.37.71.wav
[15/1005] Processing: p10085.LC.38.160.wav
[16/1005] Processing: p10085.LC.4.76.wav
[17/1005] Processing: p10085.LC.40.174.wav
[18/1005] Processing: p10085.LC.41.99999.wav
[19/1005] Processing: p10085.LC.5.134.wav
[20/1005] Processing: p10085.LC.6.99999.wav
[21/1005] Processing: p10085.LC.9.55.wav
[22/1005] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,30.69632,0.263408,26.79142,33.90816,35.34396,...,-0.051319,-0.003878,0.117989,3.759399,3.065134,0.090000,0.043012,0.191111,0.162716,-33.63273
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56685,0.143079,35.00103,35.37496,38.83751,...,-0.051151,0.000672,0.113700,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-32.83780
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.37303,0.038734,33.31597,34.01617,35.35698,...,-0.055448,-0.000904,0.110642,3.083700,2.252252,0.136000,0.094149,0.238333,0.379448,-33.32463
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,29.84379,0.294935,14.94163,34.11031,36.57949,...,-0.043690,-0.003180,0.111289,4.464286,4.566210,0.085000,0.065000,0.125555,0.134504,-34.15696
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,32.33354,0.226291,32.81301,33.89858,37.18312,...,-0.053298,-0.000288,0.106116,2.966102,3.463203,0.087500,0.047368,0.178750,0.257849,-32.63171


OpenSMILE, intensity 6dB

In [22]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations_pain" / "+6dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_6dB_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_6dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1005
[1/1005] Processing: p10085.LC.1.161.wav
[2/1005] Processing: p10085.LC.12.59.wav
[3/1005] Processing: p10085.LC.14.45.wav
[4/1005] Processing: p10085.LC.18.41.wav
[5/1005] Processing: p10085.LC.2.106.wav
[6/1005] Processing: p10085.LC.23.156.wav
[7/1005] Processing: p10085.LC.24.172.wav
[8/1005] Processing: p10085.LC.26.99999.wav
[9/1005] Processing: p10085.LC.3.60.wav
[10/1005] Processing: p10085.LC.30.132.wav
[11/1005] Processing: p10085.LC.31.99999.wav
[12/1005] Processing: p10085.LC.33.6.wav
[13/1005] Processing: p10085.LC.34.122.wav
[14/1005] Processing: p10085.LC.37.71.wav
[15/1005] Processing: p10085.LC.38.160.wav
[16/1005] Processing: p10085.LC.4.76.wav
[17/1005] Processing: p10085.LC.40.174.wav
[18/1005] Processing: p10085.LC.41.99999.wav
[19/1005] Processing: p10085.LC.5.134.wav
[20/1005] Processing: p10085.LC.6.99999.wav
[21/1005] Processing: p10085.LC.9.55.wav
[22/1005] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,30.69635,0.263404,26.79143,33.90816,35.34415,...,-0.053734,-0.005464,0.166707,3.759399,3.065134,0.090000,0.043012,0.191111,0.162716,-30.63153
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56689,0.143087,35.00103,35.37495,38.83751,...,-0.053561,-0.000992,0.160646,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-29.83669
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.36367,0.038552,33.35046,34.00359,35.33626,...,-0.057934,-0.002568,0.156898,3.083700,2.252252,0.138000,0.091957,0.236667,0.375751,-30.32344
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,29.84393,0.294924,14.94218,34.11016,36.57948,...,-0.046096,-0.004819,0.157232,4.464286,4.566210,0.085000,0.065000,0.125555,0.134504,-31.15558
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,32.33355,0.226291,32.81295,33.89860,37.18318,...,-0.055687,-0.001900,0.149933,2.966102,3.463203,0.087500,0.047368,0.178750,0.257849,-29.63066


OpenSMILE, intensity -3dB

In [23]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations_pain" / "-3dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_-3dB_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_-3dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1005
[1/1005] Processing: p10085.LC.1.161.wav
[2/1005] Processing: p10085.LC.12.59.wav
[3/1005] Processing: p10085.LC.14.45.wav
[4/1005] Processing: p10085.LC.18.41.wav
[5/1005] Processing: p10085.LC.2.106.wav
[6/1005] Processing: p10085.LC.23.156.wav
[7/1005] Processing: p10085.LC.24.172.wav
[8/1005] Processing: p10085.LC.26.99999.wav
[9/1005] Processing: p10085.LC.3.60.wav
[10/1005] Processing: p10085.LC.30.132.wav
[11/1005] Processing: p10085.LC.31.99999.wav
[12/1005] Processing: p10085.LC.33.6.wav
[13/1005] Processing: p10085.LC.34.122.wav
[14/1005] Processing: p10085.LC.37.71.wav
[15/1005] Processing: p10085.LC.38.160.wav
[16/1005] Processing: p10085.LC.4.76.wav
[17/1005] Processing: p10085.LC.40.174.wav
[18/1005] Processing: p10085.LC.41.99999.wav
[19/1005] Processing: p10085.LC.5.134.wav
[20/1005] Processing: p10085.LC.6.99999.wav
[21/1005] Processing: p10085.LC.9.55.wav
[22/1005] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,30.69629,0.263410,26.79101,33.90815,35.34415,...,-0.046522,-0.000581,0.059085,3.759399,3.065134,0.090000,0.043012,0.191111,0.162716,-39.63684
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56824,0.142929,35.00096,35.37488,38.83756,...,-0.046318,0.003799,0.056944,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-38.84159
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.36366,0.038550,33.35046,34.00276,35.33625,...,-0.050708,0.002426,0.055607,3.083700,2.252252,0.138000,0.091957,0.236667,0.375751,-39.32865
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,29.84385,0.294931,14.93992,34.11039,36.57951,...,-0.038864,0.000148,0.055731,4.464286,4.566210,0.085000,0.065000,0.125555,0.134504,-40.16165
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,32.33356,0.226291,32.81295,33.89858,37.18306,...,-0.048493,0.003067,0.053136,2.966102,3.463203,0.087500,0.047368,0.178750,0.257849,-38.63521


OpenSMILE, intensity -6dB

In [24]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations_pain" / "-6dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_-6dB_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_-6dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1005
[1/1005] Processing: p10085.LC.1.161.wav
[2/1005] Processing: p10085.LC.12.59.wav
[3/1005] Processing: p10085.LC.14.45.wav
[4/1005] Processing: p10085.LC.18.41.wav
[5/1005] Processing: p10085.LC.2.106.wav
[6/1005] Processing: p10085.LC.23.156.wav
[7/1005] Processing: p10085.LC.24.172.wav
[8/1005] Processing: p10085.LC.26.99999.wav
[9/1005] Processing: p10085.LC.3.60.wav
[10/1005] Processing: p10085.LC.30.132.wav
[11/1005] Processing: p10085.LC.31.99999.wav
[12/1005] Processing: p10085.LC.33.6.wav
[13/1005] Processing: p10085.LC.34.122.wav
[14/1005] Processing: p10085.LC.37.71.wav
[15/1005] Processing: p10085.LC.38.160.wav
[16/1005] Processing: p10085.LC.4.76.wav
[17/1005] Processing: p10085.LC.40.174.wav
[18/1005] Processing: p10085.LC.41.99999.wav
[19/1005] Processing: p10085.LC.5.134.wav
[20/1005] Processing: p10085.LC.6.99999.wav
[21/1005] Processing: p10085.LC.9.55.wav
[22/1005] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,30.69626,0.263414,26.79136,33.90815,35.34398,...,-0.044108,0.001074,0.041798,3.759399,3.065134,0.090000,0.043012,0.191111,0.162716,-42.64029
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56696,0.143081,35.00095,35.37488,38.83761,...,-0.043940,0.005591,0.040283,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-41.84466
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.36375,0.038552,33.35062,34.00272,35.33639,...,-0.048299,0.003931,0.039338,3.083700,2.252252,0.138000,0.091957,0.236667,0.375751,-42.33194
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,29.84339,0.294958,14.93551,34.11028,36.57950,...,-0.036482,0.001783,0.039428,4.464286,4.566210,0.085000,0.065000,0.125555,0.134504,-43.16550
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,32.33361,0.226287,32.81301,33.89865,37.18299,...,-0.046075,0.004801,0.037588,2.966102,3.463203,0.087500,0.047368,0.178750,0.257849,-41.63815
